In [ ]:
!git clone https://github.com/pranceraz/DeepHybrid.git
import os
os.chdir('/content/DeepHybrid')
!pwd
!pip install rl4co[graph] torch-geometric

In [ ]:
# ===============================
# COLAB READY DEEPACO TRAINING
# ===============================

import torch
import lightning as L

from generator import MyJSSPGenerator
from myenv import OperationSelectionEnv
from init_embedding import JSSPInitEmbedding, JsspEdgeEmbedding
from aco_class import MyAntSystem

from rl4co.models.zoo.nargnn.encoder import NARGNNEncoder
from rl4co.models.zoo.deepaco.policy import DeepACOPolicy
from rl4co.models.zoo.deepaco.model import DeepACO
from rl4co.models.rl.common.base import RL4COLitModule
from rl4co.utils.trainer import RL4COTrainer


In [ ]:

# ===============================
# CONFIG
# ===============================

NUM_JOBS = 6
NUM_MACHINES = 6
EMBED_DIM = 256
BATCH_SIZE = 16
VAL_BATCH_SIZE = 16
TRAIN_EPOCHS = 30
LR = 1e-4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


# ===============================
# GENERATOR + ENV
# ===============================

generator = MyJSSPGenerator(
    num_jobs=NUM_JOBS,
    num_machines=NUM_MACHINES
)

env = OperationSelectionEnv(generator, device = DEVICE)


# ===============================
# ENCODER
# ===============================

init_emb = JSSPInitEmbedding(
    embed_dim=EMBED_DIM,
    num_machines=NUM_MACHINES
)

edge_emb = JsspEdgeEmbedding(embed_dim=EMBED_DIM)

encoder = NARGNNEncoder(
    embed_dim=EMBED_DIM,
    init_embedding=init_emb,
    edge_embedding=edge_emb
)


# ===============================
# POLICY
# ===============================

policy = DeepACOPolicy(
    encoder=encoder,
    env_name="tsp",  # symbolic
    n_ants=dict(train=10, val=20, test=50),
    n_iterations=dict(train=1, val=5, test=10),
    aco_class=MyAntSystem
)


# ===============================
# LIGHTNING MODULE
# ===============================

# model = RL4COLitModule(
#     policy=policy,
#     env=env,
#     batch_size=BATCH_SIZE,
#     val_batch_size=VAL_BATCH_SIZE,
#     optimizer_kwargs=dict(lr=LR),
# )

model = DeepACO(
    env=env,
    policy=policy,
    train_with_local_search=False,
    # optimizer_kwargs=dict(lr=1e-4),
)

# ===============================
# TRAINER
# ===============================

trainer = RL4COTrainer(
    max_epochs=TRAIN_EPOCHS,
    accelerator="gpu" if DEVICE == "cuda" else "cpu",
    devices=1,
    log_every_n_steps=10,

)

trainer.fit(model)
torch.save(model.state_dict(), "deepaco_jssp_6x6.pt")

In [ ]:
torch.save(model.state_dict(), "deepaco_jssp_6x6_trained_2026_3_24.pt")